In [1]:
using JLD2
using Printf
using StaticArrays
using Statistics
using Test
using WaveGreen2D.Chebyshev: ChebyshevSeries, TransformedChebyshevSeries, gradient, hessian, contains

In [2]:
@load "../src/chebyshev_series.jld2" L₁_series L₂_series;

In [3]:
function u(x::SVector{3,Float64})
    A, B, H = x

    return SA[A, B, log(H)]
end


function ∇u(x::SVector{3,Float64})
    _, _, H = x

    grad_u = one(MMatrix{3, 3, Float64})
    grad_u[3, 3] = 1/H
    
    return SMatrix(grad_u)
end


function Hu(x::SVector{3,Float64})
    _, _, H = x

    hess_u = zero(MArray{Tuple{3, 3, 3}, Float64, 3})
    hess_u[3, 3, 3] = -1/H^2

    return SArray(hess_u)
end;

In [4]:
C₁1 = L₁_series[1]
C₁2 = TransformedChebyshevSeries(L₁_series[2], u, ∇u, Hu)
C₁3 = TransformedChebyshevSeries(L₁_series[3], u, ∇u, Hu)

C₂1 = TransformedChebyshevSeries(L₂_series[1], u, ∇u, Hu)
C₂2 = TransformedChebyshevSeries(L₂_series[2], u, ∇u, Hu)
C₂3 = TransformedChebyshevSeries(L₂_series[3], u, ∇u, Hu)
C₂4 = TransformedChebyshevSeries(L₂_series[4], u, ∇u, Hu)

function C₁(x::SVector{3, Float64})
    if contains(C₁1, x)
        return C₁1(x)
    elseif contains(C₁2, x)
        return C₁2(x)
    elseif contains(C₁3, x)
        return C₁3(x)
    else
        throw(DomainError(x))
    end
end


function C₁_gradient(x::SVector{3, Float64})
    if contains(C₁1, x)
        return gradient(C₁1, x)
    elseif contains(C₁2, x)
        return gradient(C₁2, x)
    elseif contains(C₁3, x)
        return gradient(C₁3, x)
    else
        throw(DomainError(x))
    end
end


function C₁_hessian(x::SVector{3, Float64})
    if contains(C₁1, x)
        return hessian(C₁1, x)
    elseif contains(C₁2, x)
        return hessian(C₁2, x)
    elseif contains(C₁3, x)
        return hessian(C₁3, x)
    else
        throw(DomainError(x))
    end
end


function C₂(x::SVector{3, Float64})
    if contains(C₂1, x)
        return C₂1(x)
    elseif contains(C₂2, x)
        return C₂2(x)
    elseif contains(C₂3, x)
        return C₂3(x)
    elseif contains(C₂4, x)
        return C₂4(x)
    else
        throw(DomainError(x))
    end
end


function C₂_gradient(x::SVector{3, Float64})
    if contains(C₂1, x)
        return gradient(C₂1, x)
    elseif contains(C₂2, x)
        return gradient(C₂2, x)
    elseif contains(C₂3, x)
        return gradient(C₂3, x)
    elseif contains(C₂4, x)
        return gradient(C₂4, x)
    else
        throw(DomainError(x))
    end
end


function C₂_hessian(x::SVector{3, Float64})
    if contains(C₂1, x)
        return hessian(C₂1, x)
    elseif contains(C₂2, x)
        return hessian(C₂2, x)
    elseif contains(C₂3, x)
        return hessian(C₂3, x)
    elseif contains(C₂4, x)
        return hessian(C₂4, x)
    else
        throw(DomainError(x))
    end
end;

In [6]:
tol_L = 10^2 * eps()
tol_∇L = 10^4 * eps()
tol_HL = 10^6 * eps();

In [7]:
@testset "L₁ Chebyshev" begin
    @load "../test/data/l1_test_data.jld2" x1 L1 ∇L1 HL1

    npoints = length(x1)
    
    C1 = Vector{Float64}(undef, npoints)
    C1g = Vector{Float64}(undef, npoints)
    C1h = Vector{Float64}(undef, npoints)
    
    ∇C1 = Matrix{Float64}(undef, npoints, 3)
    ∇C1h = Matrix{Float64}(undef, npoints, 3)
    
    HC1 = Array{Float64, 3}(undef, npoints, 3, 3)

    for i in 1:npoints
        y₁ = C₁(x1[i])
        y₂, ∇y₂ = C₁_gradient(x1[i])
        y₃, ∇y₃, Hy₃ = C₁_hessian(x1[i])
    
        C1[i] = y₁
        C1g[i] = y₂
        C1h[i] = y₃
        ∇C1[i, :] .= ∇y₂
        ∇C1h[i, :] .= ∇y₃
        HC1[i, :, :] .= Hy₃
    end

    # Test if ChebyshevSeries, gradient and hessian return the same value of L₁
    @test C1 == C1g
    @test C1 == C1h

    # Test if gradient and hessian return the same value of ∇L₁
    @test ∇C1 == ∇C1h

    @test all(isapprox.(L1, C1; rtol=tol_L, atol=tol_L))
    @test all(isapprox.(∇L1, ∇C1; rtol=tol_∇L, atol=tol_∇L))
    @test all(isapprox.(HL1, HC1; rtol=tol_HL, atol=tol_HL))
end

Test Summary: | Pass  Total  Time
L₁ Chebyshev  |    6      6  8.4s


Test.DefaultTestSet("L₁ Chebyshev", Any[], 6, false, false, true, 1.782325593430538e9, 1.782325601793092e9, false, "In[7]", Random.Xoshiro(0xcf472b10b95f8e61, 0x89133bd8ee98d2f4, 0x9f1f89be3af83969, 0x6217d4e58df07103, 0x34c561998d963d5f))

In [8]:
@testset "L₂ Chebyshev" begin
    @load "../test/data/l2_test_data.jld2" x2 L2 ∇L2 HL2

    npoints = length(x2)
    
    C2 = Vector{Float64}(undef, npoints)
    C2g = Vector{Float64}(undef, npoints)
    C2h = Vector{Float64}(undef, npoints)
    
    ∇C2 = Matrix{Float64}(undef, npoints, 3)
    ∇C2h = Matrix{Float64}(undef, npoints, 3)
    
    HC2 = Array{Float64, 3}(undef, npoints, 3, 3)

    for i in 1:npoints
        y₁ = C₂(x2[i])
        y₂, ∇y₂ = C₂_gradient(x2[i])
        y₃, ∇y₃, Hy₃ = C₂_hessian(x2[i])
    
        C2[i] = y₁
        C2g[i] = y₂
        C2h[i] = y₃
        ∇C2[i, :] .= ∇y₂
        ∇C2h[i, :] .= ∇y₃
        HC2[i, :, :] .= Hy₃
    end

    # Test if ChebyshevSeries, gradient and hessian return the same value of L₁
    @test C2 == C2g
    @test C2 == C2h

    # Test if gradient and hessian return the same value of ∇L₁
    @test ∇C2 == ∇C2h

    @test all(isapprox.(L2, C2; rtol=tol_L, atol=tol_L))
    @test all(isapprox.(∇L2, ∇C2; rtol=tol_∇L, atol=tol_∇L))
    @test all(isapprox.(HL2, HC2; rtol=tol_HL, atol=tol_HL))
end

Test Summary: | Pass  Total  Time
L₂ Chebyshev  |    6      6  0.2s


Test.DefaultTestSet("L₂ Chebyshev", Any[], 6, false, false, true, 1.782325602176509e9, 1.782325602334543e9, false, "In[8]", Random.Xoshiro(0xcf472b10b95f8e61, 0x89133bd8ee98d2f4, 0x9f1f89be3af83969, 0x6217d4e58df07103, 0x34c561998d963d5f))